# 05 — Components

Reusable composite structures with `@component`.

**What you learn:**
- `@component`: decorator for handlers with a body
- `main_tag='div'`: declares the DOM tag for parent validation
- Reuse: call the component multiple times

**Prerequisites:** 04_pointers

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML

from genro_bag.resolvers import FileResolver
from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager
from genro_toolbox import set_sync

set_sync()

In [ ]:
class CatalogBuilder(HtmlBuilder):
    @component(main_tag='div')
    def product_card(self, comp, name=None, price=None, description=None, **kwargs):
        card = comp.div(_class='card')
        card.h3(name or 'Unnamed')
        card.p(price or '', _class='price')
        card.p(description or '')

In [ ]:
class ProductCatalog(ReactiveManager):
    def on_init(self):
        self.page = self.register_builder('page', CatalogBuilder)

    def main(self, source):
        # FileResolver loads style.css lazily at render time (pull model)
        source.head().style(FileResolver('style.css'))
        body = source.body()
        body.h1('Product Catalog')
        catalog = body.div(_class='catalog')
        for product in self.local_store()['products']:
            catalog.product_card(**product)

In [ ]:
app = ProductCatalog()
# products.json is a plain list of dicts — loaded with json.loads.
# The for loop in main() iterates and passes each dict as **kwargs.
app.local_store('page')['products'] = json.loads(Path('products.json').read_text())
app.run()
HTML(app.page.render())